# 4. 장애 workflow 실패 테스트

**시나리오:** 빈 명령 목록과 잘못된 severity가 각각 domain guard와 API validation에서 닫히는지 확인합니다.

**학습 목표:** 실패 위치별 기대 상태, fail-closed, `executed_commands=[]` 불변조건을 검증합니다.

## 중요 변수·함수

- 빈 `proposals`: commander가 유효한 계획을 만들지 못한 실패입니다.
- `IncidentRequest.severity`: `SEV1/SEV2/SEV3`만 허용합니다.
- `create_app()`: domain workflow 이전의 HTTP 검증 경계입니다.

In [ ]:
# 이 학습 Notebook은 외부 API/DB를 사용하지 않는 fixture 모드로 고정합니다.
import os
os.environ["APP_MODE"] = "fixture"

# Notebook 위치에서 실행해도 repository의 canonical app을 가져옵니다.
from pathlib import Path
import sys

_repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()), Path.cwd())
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

# 빈 proposal을 반환하는 Agent를 주입해 domain 실패를 재현합니다.
from dataclasses import replace
from week3.app import IncidentRequest, create_app, create_fixture_services, run_incident_response

services = replace(create_fixture_services(), command=lambda incident, findings, revised: [])
blocked = run_incident_response(IncidentRequest(service='checkout', summary='Errors after deployment', severity='SEV1'), services)
assert blocked['status'] == 'blocked_manual_review'
assert blocked['executed_commands'] == []

In [ ]:
# 잘못된 severity는 FastAPI/Pydantic 경계에서 422로 거부됩니다.
from fastapi.testclient import TestClient

client = TestClient(create_app(create_fixture_services()))
response = client.post('/incidents/respond', json={'service': 'checkout', 'summary': 'errors', 'severity': 'SEV0'})
assert response.status_code == 422
{'domain_status': blocked['status'], 'api_status': response.status_code}

## 예측 과제와 해석

**예측 과제:** model/DB 예외를 fixture 성공 응답으로 바꾸면 안 되는 이유를 적으세요.

**해석:** 실패를 그럴듯한 성공으로 숨기면 운영자가 잘못된 판단을 합니다. 각 경계는 실패를 명시적으로 드러내고 실행은 항상 비어 있어야 합니다.